In [ ]:
!kaggle datasets download sovitrath/diabetic-retinopathy-224x224-2019-data
!unzip -q diabetic-retinopathy-224x224-2019-data.zip -d data
!find data -maxdepth 3

In [ ]:
import os

for root, dirs, files in os.walk("data"):
    print(root)
    if len(dirs) > 0:
        print("Folders:", dirs)
    if len(files) > 0:
        print("Files:", files[:5])   # Show first 5 files only

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow Version:", tf.__version__)

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20,
    rotation_range=15,
    zoom_range=0.15,
    horizontal_flip=True
)

valid_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.20
)

In [ ]:
train_data = train_datagen.flow_from_directory(
    directory="data/colored_images",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True
)

valid_data = valid_datagen.flow_from_directory(
    directory="data/colored_images",
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False
)

print("\nClass Indices:")
print(train_data.class_indices)

In [ ]:
# Display some sample images from the training set

images, labels = next(train_data)

class_names = list(train_data.class_indices.keys())

plt.figure(figsize=(10,10))

for i in range(9):
    plt.subplot(3,3,i+1)
    plt.imshow(images[i])
    plt.title(class_names[np.argmax(labels[i])])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)

base_model.trainable = False

model = Sequential([
    base_model,
    GlobalAveragePooling2D(),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(5, activation="softmax")
])

model.summary()

In [ ]:
model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

In [ ]:
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    verbose=1
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=10,
    callbacks=[early_stop]
)

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history["accuracy"], label="Training Accuracy")
plt.plot(history.history["val_accuracy"], label="Validation Accuracy")

plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

In [ ]:
plt.figure(figsize=(8,5))

plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")

plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.show()

In [ ]:
loss, accuracy = model.evaluate(valid_data)

print(f"\nValidation Accuracy : {accuracy*100:.2f}%")
print(f"Validation Loss     : {loss:.4f}")

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Reset generator
valid_data.reset()

# Predict
predictions = model.predict(valid_data, verbose=1)

predicted_classes = np.argmax(predictions, axis=1)
true_classes = valid_data.classes

class_names = list(valid_data.class_indices.keys())

print(classification_report(
    true_classes,
    predicted_classes,
    target_names=class_names
))

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

cm = confusion_matrix(true_classes, predicted_classes)

plt.figure(figsize=(7,6))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(class_names))

plt.xticks(tick_marks, class_names, rotation=45)
plt.yticks(tick_marks, class_names)

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(
            j, i,
            cm[i, j],
            ha="center",
            va="center",
            color="white" if cm[i, j] > cm.max()/2 else "black"
        )

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

loss, accuracy = model.evaluate(valid_data, verbose=0)

summary = pd.DataFrame({
    "Metric":[
        "Training Accuracy",
        "Validation Accuracy",
        "Validation Loss",
        "Epochs",
        "Model"
    ],
    "Value":[
        f"{history.history['accuracy'][-1]*100:.2f}%",
        f"{accuracy*100:.2f}%",
        f"{loss:.4f}",
        len(history.history["accuracy"]),
        "MobileNetV2"
    ]
})

summary

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import os

class_names = list(valid_data.class_indices.keys())

file_paths = valid_data.filepaths

random_indices = random.sample(range(len(file_paths)), 9)

plt.figure(figsize=(8, 8))

for i, idx in enumerate(random_indices):

    img = tf.keras.utils.load_img(file_paths[idx], target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img) / 255.0
    img_input = np.expand_dims(img_array, axis=0)

    prediction = model.predict(img_input, verbose=0)
    pred_index = np.argmax(prediction)
    pred_class = class_names[pred_index]
    confidence = np.max(prediction) * 100

    actual_class = os.path.basename(os.path.dirname(file_paths[idx]))

    if pred_class == actual_class:
        title_color = "green"
    else:
        title_color = "red"

    plt.subplot(3, 3, i + 1)
    plt.imshow(img)
    plt.title(
        f"Actual: {actual_class}\nPredicted: {pred_class}\nConfidence: {confidence:.1f}%",
        color=title_color,
        fontsize=11,
        fontweight="bold"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:

model.save("diabetic_retinopathy_model.h5")

print("Model saved successfully as diabetic_retinopathy_model.h5")